This uses the template to graph U1, 4 and 5 unemployment measures 

In [1]:
import datetime as dt
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd

2. We'll get data using the fredapi package. Set the path to the text file with your API key

In [2]:
from fredapi import Fred

API_KEY_PATH = "fred_api_key.txt" 
fred = Fred(api_key_file = API_KEY_PATH) 

3. Set the fed_2025 template as default

In [3]:
import graph_templates

pio.templates.default = 'fed_2025'

# Call the graph the exact same thing as its notebook (minus the ipynb suffix)
GRAPH_NAME = "copy"

# Now is a good time to set the path to the graph output folder!
GRAPH_OUTPUT_PATH = "../graph_output"


4. Use the fredapi to get the data and prepare it for graphing. Documentation on the optional parameters that can be passed to the get_series called are found here (the documentation in fredapi is out of date). 

https://fred.stlouisfed.org/docs/api/fred/series_observations.html#Description

If you get data from somewhere else thats fine too! Put the raw csv in the "raw_data" folder and read it in here. Make sure not to edit the raw data, just transform and graph it.

In [4]:
# The get_series_info method may be useful
fred.get_series_info(series_id="CPIAUCSL")

# It's good practice to store the series codes in a dictionary with their names
series_codes = {
    "U1 Unemployment Rate": "U1RATE",
    "U4 Unemployment Rate": "U4RATE",
    "U5 Unemployment Rate": "U5RATE",
}

In [5]:
today = dt.date.today()

u1_unempoyment = fred.get_series(
   series_id=series_codes["U1 Unemployment Rate"],
    observation_start=dt.date(2000, 1, 1),
    observation_end=today, 
).rename("U1 Unemployment Rate")

u4_unempoyment = fred.get_series(
   series_id=series_codes["U4 Unemployment Rate"],
    observation_start=dt.date(2000, 1, 1),
    observation_end=today, 
).rename("U4 Unemployment Rate")

u5_unempoyment = fred.get_series(
   series_id=series_codes["U5 Unemployment Rate"],
    observation_start=dt.date(2000, 1, 1),
    observation_end=today, 
).rename("U5 Unemployment Rate")

joined_df = pd.concat(
    [u1_unempoyment, u4_unempoyment, u5_unempoyment], # srs to join
    axis=1, # multiple observations per index entry
    join='inner' # only include observations where none of the 3 are missing
)

# Naming the index makes it easier tot title
joined_df.index.name = "Date"

joined_df.tail()

,U1 Unemployment Rate,U4 Unemployment Rate,U5 Unemployment Rate
Date,,,
2025-04-01,1.6,4.4,5.1
2025-05-01,1.5,4.5,5.1
2025-06-01,1.6,4.5,5.1
2025-07-01,1.8,4.5,5.2
2025-08-01,1.7,4.6,5.3


5. Now that all our data is ready, make the graph and have it save itself as a .html file to graph_output whenver the notebooks is rerun. The name of the file should exactly match the notebook name. For instance, this file "example.ipynb" produces the graph "example.html." Nice work, you made a graph! 

In [6]:
# First make the figure
fig = go.Figure()


# Loop the columns of the dataframe and plot each as a separate trace
for col in joined_df.columns:
    fig.add_trace(
        go.Scatter(
            x=joined_df.index,
            y=joined_df[col],
            mode='lines',
            name=col
        )
    )

# Update the titles, using the html tage <sup> for a subtitle 
fig.update_layout(
    title = dict(text = 'Unemployment Measures <br><sup>Monthly </sup>'),
    xaxis_title="Date",
    yaxis_title="Percent",
)

# This is graph specific, but here we want the y-axis to be percent signs 
fig.update_yaxes(
    tickformat=".2f%",
    ticksuffix="%"
)

# Again, graph specific, we have a mutliyear series and want tick marks to be years
fig.update_xaxes(
    type='date',
    tickformat='%Y',
)

# Show our figure (Dimensions may be off on different screen sizes)
fig.show()

# This should be the same for EVERY GRAPH!
# Save it to the graph_output folder with the name matching the file, as HTML
fig.write_html(GRAPH_OUTPUT_PATH + f"/{GRAPH_NAME}.html")